In [1]:
import math
import random
import copy
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict


random.seed(42)
np.random.seed(42)



Import del dataset

In [4]:
print("\n[1/6] Importazione grafo")
G = nx.read_edgelist("Dataset/facebook_combined.txt")
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
tutti_i_gradi = [grado for nodo, grado in G.degree()]
grado_massimo = max(tutti_i_gradi)
grado_minimo = min(tutti_i_gradi)
print(f"      Nodi: {n_nodes}, Archi: {n_edges}")
print(f"      Grado medio: {2*n_edges/n_nodes:.2f}")
print(f"      Grado massimo: {grado_massimo}")
print(f"      Grado minimo: {grado_minimo}")
print(f"      Clustering medio: {nx.average_clustering(G):.4f}")



[1/6] Importazione grafo
      Nodi: 4039, Archi: 88234
      Grado medio: 43.69
      Grado massimo: 1045
      Grado minimo: 1
      Clustering medio: 0.6055


Visualizza grafo

In [3]:
"""Visualizza proprietà del grafo."""
out_path = "/Users/giuseppepiosorrentino/RetiSocialiProj/Results"
degrees = [d for _, d in G.degree()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribuzione dei gradi", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Grado", fontsize=11)
axes[0].set_ylabel("Frequenza", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Draw small version of the graph
pos = nx.spring_layout(G, seed=42, k=0.5)
nx.draw_networkx(G, pos=pos, ax=axes[1], node_size=20, node_color="#2196F3",
                   edge_color="#BBBBBB", with_labels=False, alpha=0.8, width=0.5)
axes[1].set_title("Visualizzazione della rete (Barabási-Albert)", fontsize=13, fontweight="bold")
axes[1].axis("off")

fig.suptitle(f"Rete: n={G.number_of_nodes()}, m={G.number_of_edges()}, "
             f"<k>={2*G.number_of_edges()/G.number_of_nodes():.1f}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Salvato: {out_path}")

  Salvato: /Users/giuseppepiosorrentino/RetiSocialiProj/Results


Inizializzazione delle Funzioni di costo

In [ ]:
import random
import math

def cost_random(G, low=1, high=5, seed=42):
    rng = random.Random(seed)
    return {v: rng.randint(low, high) for v in G.nodes()}

def cost_half_degree(G):
    return {v: max(1, math.ceil(G.degree(v) / 2)) for v in G.nodes()}

#VALUTIAMO, è OPZIONALE
def cost_custom(G):
    """Costo basato su betweenness centrality: 
    i nodi ponte tra comunità costano di più"""
    bet = nx.betweenness_centrality(G)
    return {v: max(1, round(1 + bet[v] * 10)) for v in G.nodes()}

# Istanzia le tre funzioni costo sul grafo G
c_random = cost_random(G)
c_half   = cost_half_degree(G)
#OPZIONALE
c_custom = cost_custom(G)

print("Esempio costi (primi 5 nodi):")
for v in list(G.nodes())[:5]:
    print(f"  nodo {v}: random={c_random[v]}, d/2={c_half[v]}, custom={c_custom[v]}")



Esempio costi (primi 5 nodi):
  nodo 0: random=1, d/2=174, custom=2
  nodo 1: random=1, d/2=9, custom=1
  nodo 2: random=3, d/2=5, custom=1
  nodo 3: random=2, d/2=9, custom=1
  nodo 4: random=2, d/2=5, custom=1


In [8]:
costo_totale_random = sum(c_random.values())
costo_totale_half   = sum(c_half.values())
costo_totale_custom = sum(c_custom.values())
print(f"Costo totale random: {costo_totale_random}")
print(f"Costo totale half:   {costo_totale_half}")
print(f"Costo totale custom: {costo_totale_custom}")

Costo totale random: 12173
Costo totale half:   89243
Costo totale custom: 4059


Introduciamo delle approssimazioni analitiche che stimano quello che la funzione obiettivo dobrebbe fare, simulare l'intera cascata per ogni nodo è costoso visto l'elevato numero di nodi nell'grafo. 
Le approssimazioni ci permettono  di stimare quanto S è efficace guardando solo la struttura locale del grafo (quanti vicini di ogni nodo sono già in S rispetto alla soglia).


In [9]:
def f1(G, S):
    val = 0
    for v in G.nodes():
        overlap = sum(1 for u in G.neighbors(v) if u in S)
        val += min(overlap, math.ceil(G.degree(v) / 2))
    return val

def f2(G, S):
    val = 0
    for v in G.nodes():
        overlap = sum(1 for u in G.neighbors(v) if u in S)
        threshold = math.ceil(G.degree(v) / 2)
        for i in range(1, overlap + 1):
            val += max(threshold - i + 1, 0)
    return val

def f3(G, S):
    val = 0
    for v in G.nodes():
        dv = G.degree(v)
        if dv == 0:
            continue
        overlap = sum(1 for u in G.neighbors(v) if u in S)
        threshold = math.ceil(dv / 2)
        for i in range(1, overlap + 1):
            denom = dv - i + 1
            if denom > 0:
                val += max((threshold - i + 1) / denom, 0)
    return val

Algoritmo di Majority Cascade

In [10]:
def majority_cascade(G, S):
    """
    Simula il processo di attivazione con regola Majority.
    Un nodo v si attiva se almeno metà dei suoi vicini sono già attivi.
    Restituisce Inf[S] = insieme di tutti i nodi attivati.
    """
    infected = set(S)
    
    while True:
        new_infected = set()
        for v in G.nodes():
            if v not in infected:
                dv = G.degree(v)
                if dv == 0:
                    continue
                neighbors_infected = sum(1 for u in G.neighbors(v) if u in infected)
                if neighbors_infected >= math.ceil(dv / 2):
                    new_infected.add(v)
        
        if not new_infected:  # nessun nuovo nodo attivato: la cascata si ferma
            break
        infected |= new_infected
    
    return infected

In [11]:
# Prova con un piccolo seed set manuale
S_test = set(list(G.nodes())[:5])  # primi 5 nodi come seed
inf_test = majority_cascade(G, S_test)

print(f"Seed set: {S_test}")
print(f"Nodi attivati: {len(inf_test)} / {G.number_of_nodes()}")

Seed set: {'0', '2', '3', '4', '1'}
Nodi attivati: 53 / 4039


ALGORITMO 1: Cost-Seeds-Greedy

In [12]:
def marginal_gain(G, v, S_current, fi_func):
    """
    Calcola Δ_v f_i(S) = f_i(S ∪ {v}) - f_i(S)
    cioè quanto migliora la stima aggiungendo il nodo v a S.
    """
    S_new = S_current | {v}
    return fi_func(G, S_new) - fi_func(G, S_current)

def cost_seeds_greedy(G, k, c, fi_func):
    """
    Ad ogni passo sceglie il nodo u con miglior rapporto
    guadagno marginale / costo, finché il budget k non viene superato.
    Restituisce S_p, l'ultimo seed set con costo <= k.
    """
    S_p = set()   # seed set precedente (sempre entro budget)
    S_d = set()   # seed set corrente (potrebbe sforare)

    while True:
        best_u = None
        best_ratio = -1

        for v in set(G.nodes()) - S_d:
            if c[v] == 0:
                continue
            ratio = marginal_gain(G, v, S_d, fi_func) / c[v]
            if ratio > best_ratio:
                best_ratio = ratio
                best_u = v

        if best_u is None:
            break

        S_p = set(S_d)        # salva lo stato precedente
        S_d = S_d | {best_u}  # aggiunge il nodo migliore

        if sum(c[u] for u in S_d) > k:
            break  # budget superato, ritorna S_p

    return S_p

In [ ]:
#costo fittizzio, andrebbe proporzionato al costo totale, DA AGGIUNGERE

k = 20
S_greedy_f1 = cost_seeds_greedy(G, k, c_random, f1)
S_greedy_f2 = cost_seeds_greedy(G, k, c_random, f2)
S_greedy_f3 = cost_seeds_greedy(G, k, c_random, f3)

print(f"Seed set Greedy-f1: {len(S_greedy_f1)} nodi, costo={sum(c_random[u] for u in S_greedy_f1)}")
print(f"Seed set Greedy-f2: {len(S_greedy_f2)} nodi, costo={sum(c_random[u] for u in S_greedy_f2)}")
print(f"Seed set Greedy-f3: {len(S_greedy_f3)} nodi, costo={sum(c_random[u] for u in S_greedy_f3)}")

print(f"\n|Inf[G, S_f1]| = {len(majority_cascade(G, S_greedy_f1))}")
print(f"|Inf[G, S_f2]| = {len(majority_cascade(G, S_greedy_f2))}")
print(f"|Inf[G, S_f3]| = {len(majority_cascade(G, S_greedy_f3))}")